# 13 — Compare all five X-IIoTID validation baselines

This notebook reads saved **validation metrics only** and generates comparison charts. It **does not train any model** and **does not access the test set**. Requires notebooks 08–12 to have run and written their metric CSV files.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path(r'D:\J26-CS-336-Network-Flow-V2-Git\network_flow_v2')
AUDIT=ROOT/'reports/audits';FIG=ROOT/'reports/figures'; FIG.mkdir(parents=True,exist_ok=True)
models=['decision_tree','random_forest','extra_trees','lightgbm','adaboost']
paths=[AUDIT/f'x_iiotid_{m}_validation_v1.csv' for m in models]
missing=[str(p) for p in paths if not p.is_file()]
if missing:
    raise FileNotFoundError('Run model notebooks 08–12 first: '+', '.join(missing))
res=pd.concat([pd.read_csv(p) for p in paths],ignore_index=True)
assert len(res)==5
assert res.frozen_manifest_sha256.nunique()==1, 'Models used DIFFERENT frozen splits!'
assert res.train_rows_used.nunique()==1, 'Models used different train sample sizes'
assert res.validation_rows.nunique()==1, 'Models used different validation sizes'
assert not res.test_evaluated.any(), 'Unexpected test evaluation in pilot metrics'
cols=['model','precision_attack','recall_attack','f1_attack','balanced_accuracy','false_positive_rate','fit_seconds','predict_ms_per_1000']
display(res[cols].round(4).sort_values('f1_attack',ascending=False))
for ylabel,metrics,filename in [
    ('Validation score', ['recall_attack','f1_attack','balanced_accuracy','false_positive_rate'],'x_iiotid_five_models_validation_scores_v1.png'),
    ('Milliseconds per 1,000 validation rows',['predict_ms_per_1000'],'x_iiotid_five_models_inference_timing_v1.png'),
    ('Training seconds',['fit_seconds'],'x_iiotid_five_models_training_timing_v1.png')]:
    fig,ax=plt.subplots(figsize=(10,4.7))
    res.set_index('model')[metrics].plot(kind='bar',ax=ax)
    ax.set_xlabel('Model');ax.set_ylabel(ylabel);ax.set_title('X-IIoTID — same frozen split, same train sample')
    ax.tick_params(axis='x',rotation=25)
    fig.tight_layout();fig.savefig(FIG/filename,dpi=170);plt.show()
res[cols+['train_rows_used','validation_rows','frozen_manifest_sha256','test_evaluated']].to_csv(AUDIT/'x_iiotid_five_models_validation_comparison_v1.csv',index=False)
print('Completed validation comparison. No test evaluation performed.')


**Interpretation:** Compare F1/recall and false-positive rate alongside laptop inference time. Selecting a model by validation performance is acceptable, but final **test data must remain unseen** until model/ensemble selection is fixed. Time measured on this computer is not representative of Raspberry Pi.
